<a href="https://colab.research.google.com/github/natalianowak1/airbnb-price-optimization/blob/main/airbnb_price_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Przygotowanie środowiska i import bibliotek



In [11]:
import pandas as pd

# 2. Wczytanie i wstępny przegląd danych

In [14]:
df = pd.read_excel("/content/sample_data/Airbnb_Open_Data.xlsx")

Podgląd pierwszych pięć wierszy

In [20]:
pd.set_option('display.max_columns', None)
print(df.head())

        id                                              NAME      host id  \
0  1001254                Clean & quiet apt home by the park  80014485718   
1  1002102                             Skylit Midtown Castle  52335172823   
2  1002403               THE VILLAGE OF HARLEM....NEW YORK !  78829239556   
3  1002755                                               NaN  85098326012   
4  1003689  Entire Apt: Spacious Studio/Loft by central park  92037596077   

  host_identity_verified host name neighbourhood group neighbourhood  \
0            unconfirmed  Madaline            Brooklyn    Kensington   
1               verified     Jenna           Manhattan       Midtown   
2                    NaN     Elise           Manhattan        Harlem   
3            unconfirmed     Garry            Brooklyn  Clinton Hill   
4               verified    Lyndon           Manhattan   East Harlem   

   latitude  longitude        country country code  instant_bookable  \
0  40.64749  -73.97237  United S

Identyfikacja zmiennych i weryfikacja typów

In [16]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102599 entries, 0 to 102598
Data columns (total 25 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102599 non-null  int64  
 1   NAME                            102349 non-null  object 
 2   host id                         102599 non-null  int64  
 3   host_identity_verified          102310 non-null  object 
 4   host name                       102193 non-null  object 
 5   neighbourhood group             102570 non-null  object 
 6   neighbourhood                   102583 non-null  object 
 7   latitude                        102591 non-null  float64
 8   longitude                       102591 non-null  float64
 9   country                         102067 non-null  object 
 10  country code                    102468 non-null  object 
 11  instant_bookable                102494 non-null  float64
 12  cancellation_pol

Zliczanie brakujących wartości (nulli) w każdej kolumnie

In [21]:
print(df.isnull().sum())

id                                     0
NAME                                 250
host id                                0
host_identity_verified               289
host name                            406
neighbourhood group                   29
neighbourhood                         16
latitude                               8
longitude                              8
country                              532
country code                         131
instant_bookable                     105
cancellation_policy                   76
room type                              0
Construction year                    214
price                                247
service fee                          247
minimum nights                       409
number of reviews                    183
reviews per month                  15879
review rate number                   326
calculated host listings count       319
availability 365                     448
house_rules                        52131
license         

Liczba zduplikowanych wierszy

In [22]:
print(df.duplicated().sum())

541


Wnioski ze wstępnej analizy:
1. Metryki ogólne:
*  Tabela zawiera 25 kolumnn o nazwach kolejno: 'id', 'NAME', 'host id', 'host_identity_verified', 'host name', 'neighbourhood group', 'neighbourhood', 'latitude', 'longitude', 'country', 'country code', 'instant_bookable', 'cancellation_policy', 'room type', 'Construction year', 'price', 'service fee', 'minimum nights', 'number of reviews', 'reviews per month', 'review rate number', 'calculated host listings count', 'availability 365', 'house_rules', 'license'
* Tabela liczy 102 599 wierszy (ogłoszeń).
* W zbiorze wykryto 541 całkowicie zduplikowanych wierszy, które należy usunąć.
* Jedyne kolumny, które są w 100% kompletne (0 nulli), to: `id`, `host id` oraz `room type`. W pozostałych 22 kolumnach występują nulle.

2. Zidentyfikowane anomalie i plan ich czyszczenia:
*   Kolumny takie jak 'price' (cena) oraz 'service fee' (opłata serwisowa) są już zapisane jako dane liczbowe (float64), co umożliwia bezpośrednie wykonywanie obliczeń i analizę statystyczną. Posiadają jednak po 247 braków danych, które trzeba uzupełnić (np. medianą cen).
* Kolumna `instant_bookable` zawiera wartości binarne (1 i 0) oraz 105 nulli. W etapie czyszczenia należy zastąpić nulle wartością `0` (bezpieczne założenie, że brak informacji oznacza brak natychmiastowej rezerwacji), a całą kolumnę zmienić na typ logiczny (`boolean`).
* Kolumna `reviews per month` zawiera aż 15 879 braków danych. Wynika to z faktu, że nowo dodane nieruchomości nie mają jeszcze żadnych opinii. Nulle w tej kolumnie zostaną zastąpione wartością `0.0`.
* Kolumna 'license' zawiera tylko 2 wypełnione wiersze, a cała reszta to nulle (kolumna kwalifikuje się do usunięcia).
* Kolumna `house_rules` zawiera informacje w mniej niż połowie wierszy (52 131 nulli). Braki zostaną zastąpione wartością domyślną "No rules specified".
* Pozostałe kolumny tekstowe i lokalizacyjne (np. `NAME`, `host name`, `neighbourhood`) posiadają niewielkie liczby braków (od kilku do kilkuset). W ich przypadku wiersze z nullami zostaną zastąpione wartością "Unknown".

